# Etapa 6 — Restos a Pagar
### Em quais funções as capitais mais empurram despesas para o exercício seguinte?

Os **Restos a Pagar (RP)** representam obrigações financeiras assumidas pelo governo em um exercício que não foram quitadas dentro do mesmo ano. Eles são inscritos como dívida para o exercício seguinte e indicam **pressão fiscal acumulada** — quanto maior, maior o risco de comprometimento de orçamentos futuros.

## 1. Conceitos Fundamentais

O Siconfi disponibiliza dois tipos de restos a pagar:

| Tipo | Descrição | O que significa |
|------|-----------|----------------|
| **RPNP** — Restos a Pagar Não Processados | Despesas empenhadas mas **não liquidadas** no ano | O serviço/produto **não foi entregue** ainda |
| **RPP** — Restos a Pagar Processados | Despesas liquidadas mas **não pagas** no ano | O serviço foi entregue, mas o **pagamento não saiu** |

Fluxo normal de uma despesa pública:
```
Empenho → Liquidação → Pagamento
  ↑ RPNP: parou aqui         ↑ RPP: parou aqui
```

**Indicadores calculados nesta análise:**

$$\%RPNP = \frac{\text{Inscrição de RPNP}}{\text{Despesas Empenhadas}} \times 100$$

$$\%RPP = \frac{\text{Inscrição de RPP}}{\text{Despesas Empenhadas}} \times 100$$

$$\%Total\ RP = \frac{\text{RPNP} + \text{RPP}}{\text{Despesas Empenhadas}} \times 100$$
$$\%Execução_{Real} = \frac{Pago}{Empenhado - RPNP - RPP} \times 100$$

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Adiciona a raiz do projeto ao sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src.banco.conexao_duckdb import conectar
from src.utils.constantes import CAMINHO_DUCKDB

pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
con = conectar(CAMINHO_DUCKDB)
print(f'Conectado a: {CAMINHO_DUCKDB}')

# Verificar quais valores de 'Coluna' existem na base para restos a pagar
print('\nValores únicos da coluna Coluna (tipo de despesa):')
for row in con.execute('SELECT DISTINCT "Coluna" FROM despesas_finbra ORDER BY 1').fetchall():
    print(f'  {row[0]}')

## 2. Restos a Pagar por Função — Visão Nacional (2020-2024)

Vamos identificar em quais funções as capitais mais acumulam restos a pagar. Usamos o período completo de 2020-2024 (excluindo 2025 que está incompleto).

In [ ]:
query_rp_funcao = """
WITH base AS (
    SELECT
        Conta as Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) as Empenhado,
        SUM(CASE WHEN Coluna = 'Inscrição de Restos a Pagar Não Processados' THEN Valor ELSE 0 END) as RPNP,
        SUM(CASE WHEN Coluna = 'Inscrição de Restos a Pagar Processados' THEN Valor ELSE 0 END) as RPP
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
    GROUP BY Conta
)
SELECT
    Funcao,
    Empenhado,
    RPNP,
    RPP,
    RPNP + RPP as Total_RP,
    CASE WHEN Empenhado > 0 THEN ROUND(RPNP / Empenhado * 100, 2) ELSE 0 END as Pct_RPNP,
    CASE WHEN Empenhado > 0 THEN ROUND(RPP / Empenhado * 100, 2) ELSE 0 END as Pct_RPP,
    CASE WHEN Empenhado > 0 THEN ROUND((RPNP + RPP) / Empenhado * 100, 2) ELSE 0 END as Pct_Total_RP
FROM base
WHERE Empenhado > 0
ORDER BY Pct_Total_RP DESC
"""

df_rp_funcao = con.execute(query_rp_funcao).df()
print(f'Resultado: {len(df_rp_funcao)} funções')
df_rp_funcao[['Funcao', 'Pct_RPNP', 'Pct_RPP', 'Pct_Total_RP']].head(10)

## 3. Restos a Pagar por Capital — Ranking Nacional (2020-2024)

Identificamos quais capitais possuem a maior proporção de despesas empenhadas que não foram pagas dentro do mesmo exercício.

In [ ]:
query_rp_capital = """
WITH base AS (
    SELECT
        Instituição,
        UF,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) as Pago,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) as Empenhado,
        SUM(CASE WHEN Coluna = 'Inscrição de Restos a Pagar Não Processados' THEN Valor ELSE 0 END) as RPNP,
        SUM(CASE WHEN Coluna = 'Inscrição de Restos a Pagar Processados' THEN Valor ELSE 0 END) as RPP
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
    GROUP BY Instituição, UF
)
SELECT
    Instituição,
    UF,
    Empenhado,
    Pago,
    RPNP,
    RPP,
    RPNP + RPP as Total_RP,
    CASE WHEN Empenhado > 0 THEN ROUND(RPNP / Empenhado * 100, 2) ELSE 0 END as Pct_RPNP,
    CASE WHEN Empenhado > 0 THEN ROUND(RPP / Empenhado * 100, 2) ELSE 0 END as Pct_RPP,
    CASE WHEN Empenhado > 0 THEN ROUND((RPNP + RPP) / Empenhado * 100, 2) ELSE 0 END as Pct_Total_RP,
    CASE WHEN (Empenhado - RPNP - RPP) > 0 THEN ROUND(Pago / (Empenhado - RPNP - RPP) * 100, 2) ELSE NULL END as Pct_Exec_Real
FROM base
WHERE Empenhado > 0
ORDER BY Pct_Total_RP DESC
"""

df_rp_capital = con.execute(query_rp_capital).df()

# Extrair nome limpo
df_rp_capital['Capital'] = df_rp_capital['Instituição'].apply(
    lambda x: x.replace('Prefeitura Municipal de ', '').replace('Prefeitura Municipal do ', '').split(' - ')[0].strip()
)

# Posição de Maceió
maceio_row = df_rp_capital[df_rp_capital['Capital'] == 'Maceió'].iloc[0]
pos_maceio = df_rp_capital.reset_index(drop=True).index[df_rp_capital['Capital'] == 'Maceió'][0] + 1

print(f'Ranking de capitais por % de Restos a Pagar Total (2020-2024)')
print(f'Posição de Maceió: {pos_maceio}º de {len(df_rp_capital)}')
print(f'Maceió — %RPNP: {maceio_row["Pct_RPNP"]:.2f}% | %RPP: {maceio_row["Pct_RPP"]:.2f}% | %Total: {maceio_row["Pct_Total_RP"]:.2f}%')
print(f'Maceió — %Execução Real: {maceio_row["Pct_Exec_Real"]:.2f}%')
df_rp_capital[['Capital', 'UF', 'Pct_RPNP', 'Pct_RPP', 'Pct_Total_RP', 'Pct_Exec_Real']]

## 4. Análise Comparativa — Goiânia vs Maceió vs Natal

Aprofundamos a análise nos 3 estudos de caso para entender em **quais funções** cada capital mais acumula restos a pagar.

In [ ]:
query_rp_casos = """
WITH base AS (
    SELECT
        Instituição,
        Conta as Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) as Pago,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) as Empenhado,
        SUM(CASE WHEN Coluna = 'Inscrição de Restos a Pagar Não Processados' THEN Valor ELSE 0 END) as RPNP,
        SUM(CASE WHEN Coluna = 'Inscrição de Restos a Pagar Processados' THEN Valor ELSE 0 END) as RPP
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
      AND (
        Instituição LIKE '%Goiânia%' OR
        Instituição LIKE '%Maceió%' OR
        Instituição LIKE '%Natal%'
      )
    GROUP BY Instituição, Conta
)
SELECT
    Instituição,
    Funcao,
    Empenhado,
    Pago,
    RPNP,
    RPP,
    RPNP + RPP as Total_RP,
    CASE WHEN Empenhado > 0 THEN ROUND(RPNP / Empenhado * 100, 2) ELSE 0 END as Pct_RPNP,
    CASE WHEN Empenhado > 0 THEN ROUND(RPP / Empenhado * 100, 2) ELSE 0 END as Pct_RPP,
    CASE WHEN Empenhado > 0 THEN ROUND((RPNP + RPP) / Empenhado * 100, 2) ELSE 0 END as Pct_Total_RP,
    CASE WHEN (Empenhado - RPNP - RPP) > 0 THEN ROUND(Pago / (Empenhado - RPNP - RPP) * 100, 2) ELSE NULL END as Pct_Exec_Real
FROM base
WHERE Empenhado > 0
ORDER BY Instituição, Pct_Total_RP DESC
"""

df_rp_casos = con.execute(query_rp_casos).df()
df_rp_casos['Capital'] = df_rp_casos['Instituição'].apply(
    lambda x: x.replace('Prefeitura Municipal de ', '').replace('Prefeitura Municipal do ', '').split(' - ')[0].strip()
)

print('=== TOP 5 FUNÇÕES COM MAIOR % DE RESTOS A PAGAR POR CAPITAL ===\n')
for capital in ['Goiânia', 'Maceió', 'Natal']:
    print(f'\n{capital.upper()}')
    df_cap = df_rp_casos[df_rp_casos['Capital'] == capital].head(5)
    print(df_cap[['Funcao', 'Pct_RPNP', 'Pct_RPP', 'Pct_Total_RP', 'Pct_Exec_Real']].to_string(index=False))

## 5. Visualizações
### 5.1 Ranking de Capitais por % de Restos a Pagar Total

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
fig, ax = plt.subplots(figsize=(12, 9))

df_plot = df_rp_capital.sort_values('Pct_Total_RP', ascending=True).copy()

# Cores: destaque para os 3 casos
cores = ['#e74c3c' if c == 'Natal' else '#3498db' if c == 'Maceió' else
         '#2ecc71' if c == 'Goiânia' else '#95a5a6'
         for c in df_plot['Capital']]

barras = ax.barh(df_plot['Capital'], df_plot['Pct_Total_RP'], color=cores, edgecolor='white')

for barra, valor in zip(barras, df_plot['Pct_Total_RP']):
    ax.text(barra.get_width() + 0.1, barra.get_y() + barra.get_height()/2,
            f'{valor:.1f}%', va='center', fontsize=9, fontweight='bold')

media_nacional = df_rp_capital['Pct_Total_RP'].mean()
ax.axvline(x=media_nacional, color='black', linestyle='--', linewidth=1.5,
           label=f'Média nacional: {media_nacional:.1f}%')

ax.set_xlabel('% do Empenhado inscrito como Restos a Pagar (%)', fontsize=11)
ax.set_title('Ranking de Capitais por % de Restos a Pagar Total\n(RPNP + RPP) / Empenhado × 100 — Período 2020-2024',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Goiânia (Melhor geral)'),
    Patch(facecolor='#3498db', label='Maceió'),
    Patch(facecolor='#e74c3c', label='Natal (Pior geral)'),
    Patch(facecolor='#95a5a6', label='Outras capitais'),
]
ax.legend(handles=legend_elements, fontsize=10, loc='lower right')

plt.tight_layout()
plt.show()

### 5.2 Composição RPNP vs RPP — Goiânia, Maceió e Natal

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

df_casos_rp = df_rp_capital[df_rp_capital['Capital'].isin(['Goiânia', 'Maceió', 'Natal'])].copy()
df_casos_rp = df_casos_rp.sort_values('Pct_Total_RP')

x = range(len(df_casos_rp))
largura = 0.35

barras_rpnp = ax.bar([i - largura/2 for i in x], df_casos_rp['Pct_RPNP'], largura,
                      label='RPNP (Não Processados)', color='#e67e22', edgecolor='white')
barras_rpp = ax.bar([i + largura/2 for i in x], df_casos_rp['Pct_RPP'], largura,
                     label='RPP (Processados)', color='#9b59b6', edgecolor='white')

for barra in barras_rpnp:
    v = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2, v + 0.1, f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
for barra in barras_rpp:
    v = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2, v + 0.1, f'{v:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(list(x))
ax.set_xticklabels(df_casos_rp['Capital'].tolist(), fontsize=12)
ax.set_ylabel('% do Empenhado (%)', fontsize=11)
ax.set_title('Composição dos Restos a Pagar\nRPNP vs RPP — Goiânia, Maceió e Natal (2020-2024)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
con.close()
print('Conexão com DuckDB encerrada.')

In [ ]:
print('✓ Notebook 06 executado com sucesso')
print(f'✓ Funções analisadas: {len(df_rp_funcao)}')
print(f'✓ Capitais no ranking: {len(df_rp_capital)}')
print(f'✓ Maceió — %Total RP: {maceio_row["Pct_Total_RP"]:.2f}% | %Exec Real: {maceio_row["Pct_Exec_Real"]:.2f}%')
print(f'✓ Data/hora: {pd.Timestamp.now()}')

## Conclusão da Etapa 6 — Restos a Pagar

### O que descobrimos:

1. **Funções mais problemáticas**: Identificamos quais áreas acumulam maior proporção de restos a pagar em nível nacional.
2. **Ranking de capitais**: Qual capital pressiona mais orçamentos futuros deixando compromissos em aberto.
3. **Comparação dos 3 casos**: Como Goiânia, Maceió e Natal se diferenciam na gestão de restos a pagar por função.

### Relação com a Taxa de Execução:

Os restos a pagar explicam a diferença entre a **Taxa de Execução** (Pago/Empenhado × 100) e os 100%:

$$\% Restante = \% RPNP + \% RPP$$

Em outras palavras:

$$Taxa\ de\ Execução + \% Total\ RP \approx 100\%$$

Uma capital com muitos restos a pagar tem taxa de execução baixa — e uma possível fragilidade no planejamento orçamentário.